# TCGA/GEO RNA-seq Analysis Template

用于公开队列下载、表达矩阵整理、临床信息合并、差异分析和单基因/基因集验证。适合接在 `RNAseq_General.ipynb` 之后作为外部验证模板。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
PROJECT_ID <- "TCGA-STAD"         # e.g. "TCGA-STAD", "TCGA-BRCA"
DATA_TYPE <- "Gene Expression Quantification"
WORKFLOW_TYPE <- "STAR - Counts"  # For modern TCGA RNA-seq counts
GENE_ID_TYPE <- "ENSEMBL"         # TCGA usually starts as Ensembl IDs

GEO_ACCESSION <- NULL              # e.g. "GSEXXXXX"; set NULL to skip GEO download
GROUP_COLUMN <- "group"            # Column in sample metadata used for design
GROUP_LEVELS <- c("Control", "Tumor")
COMPARISONS <- list(c("Tumor_vs_Control", "Tumor", "Control"))

MIN_COUNT <- 10
PADJ_THRESH <- 0.05
LOG2FC_THRESH <- 1
DESIGN_FORMULA <- ~ group

TARGET_GENES <- c("CD274", "PDCD1", "CTLA4", "CXCL9", "CXCL10")

OUTDIR <- "TCGA_GEO_Template_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)
cat("Configuration complete. Output:", OUTDIR, "\n")


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("TCGAbiolinks", "GEOquery", "DESeq2", "SummarizedExperiment", "AnnotationDbi", "org.Hs.eg.db", "clusterProfiler", "EnhancedVolcano"))
# install.packages(c("tidyverse", "pheatmap", "survminer", "ashr"))

suppressPackageStartupMessages({
  library(TCGAbiolinks)
  library(GEOquery)
  library(SummarizedExperiment)
  library(DESeq2)
  library(AnnotationDbi)
  library(org.Hs.eg.db)
  library(clusterProfiler)
  library(EnhancedVolcano)
  library(tidyverse)
  library(pheatmap)
  library(ashr)
})

theme_publication <- function(base_size = 12) {
  theme_bw(base_size = base_size) +
    theme(
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.7),
      axis.text = element_text(color = "black"),
      axis.title = element_text(face = "bold"),
      plot.title = element_text(face = "bold", hjust = 0.5),
      legend.title = element_text(face = "bold")
    )
}
theme_set(theme_publication())


## 3. TCGA Download and Preparation

In [ ]:
query <- GDCquery(
  project = PROJECT_ID,
  data.category = "Transcriptome Profiling",
  data.type = DATA_TYPE,
  workflow.type = WORKFLOW_TYPE
)
GDCdownload(query)
tcga_se <- GDCprepare(query)
saveRDS(tcga_se, file.path(OUTDIR, paste0(PROJECT_ID, "_SummarizedExperiment.rds")))

count_mat <- assay(tcga_se)
metadata <- as.data.frame(colData(tcga_se))
metadata$sample <- colnames(count_mat)

cat("TCGA count matrix:", nrow(count_mat), "genes x", ncol(count_mat), "samples\n")
write.csv(metadata, file.path(OUTDIR, "TCGA_sample_metadata.csv"), row.names = FALSE)


## 4. Gene Annotation and Matrix Cleaning

In [ ]:
clean_ensembl <- function(x) sub("\\..*$", "", x)
ensembl_ids <- clean_ensembl(rownames(count_mat))
map <- AnnotationDbi::select(org.Hs.eg.db, keys = unique(ensembl_ids), keytype = "ENSEMBL", columns = c("SYMBOL", "ENTREZID"))
map <- map[!is.na(map$SYMBOL) & map$SYMBOL != "", ]
map <- map[!duplicated(map$ENSEMBL), ]

anno <- data.frame(ENSEMBL = ensembl_ids, row_id = rownames(count_mat)) %>% left_join(map, by = "ENSEMBL")
keep <- !is.na(anno$SYMBOL)
count_symbol <- as.data.frame(count_mat[keep, , drop = FALSE])
count_symbol$SYMBOL <- anno$SYMBOL[keep]
count_symbol <- count_symbol %>% group_by(SYMBOL) %>% summarise(across(everything(), sum), .groups = "drop")
count_mat_symbol <- as.matrix(count_symbol[, -1])
rownames(count_mat_symbol) <- count_symbol$SYMBOL

write.csv(count_mat_symbol, file.path(OUTDIR, "TCGA_counts_SYMBOL.csv"))
cat("Symbol matrix:", nrow(count_mat_symbol), "genes x", ncol(count_mat_symbol), "samples\n")


## 5. Clinical and Group Annotation

In [ ]:
clinical <- GDCquery_clinic(project = PROJECT_ID, type = "clinical")
write.csv(clinical, file.path(OUTDIR, "TCGA_clinical.csv"), row.names = FALSE)

# Edit this block for project-specific grouping.
# Example: tumor/normal from TCGA sample type.
sample_type <- metadata$shortLetterCode
metadata$group <- ifelse(sample_type == "TP", "Tumor", ifelse(sample_type == "NT", "Control", NA))
metadata <- metadata[!is.na(metadata$group), ]
count_mat_symbol <- count_mat_symbol[, metadata$sample, drop = FALSE]
metadata$group <- factor(metadata$group, levels = GROUP_LEVELS)

stopifnot(all(colnames(count_mat_symbol) == metadata$sample))
print(table(metadata$group, useNA = "ifany"))


## 6. DESeq2 Differential Expression

In [ ]:
keep <- rowSums(count_mat_symbol >= MIN_COUNT) >= min(table(metadata$group))
dds <- DESeqDataSetFromMatrix(
  countData = round(count_mat_symbol[keep, ]),
  colData = metadata,
  design = DESIGN_FORMULA
)
dds <- DESeq(dds)
vsd <- vst(dds, blind = TRUE)

res_list <- list()
for (comp in COMPARISONS) {
  comp_name <- comp[1]; treat <- comp[2]; ctrl <- comp[3]
  res <- results(dds, contrast = c(GROUP_COLUMN, treat, ctrl), alpha = PADJ_THRESH)
  res <- lfcShrink(dds, contrast = c(GROUP_COLUMN, treat, ctrl), res = res, type = "ashr")
  df <- as.data.frame(res)
  df$gene <- rownames(df)
  df$significance <- ifelse(df$padj < PADJ_THRESH & abs(df$log2FoldChange) > LOG2FC_THRESH,
                            ifelse(df$log2FoldChange > 0, "Up", "Down"), "Not_Sig")
  res_list[[comp_name]] <- df
  write.csv(df, file.path(OUTDIR, paste0("DEG_", comp_name, ".csv")), row.names = FALSE)
}
saveRDS(list(dds = dds, vsd = vsd, res_list = res_list), file.path(OUTDIR, "DESeq2_results.rds"))


## 7. QC and Visualization

In [ ]:
pca <- plotPCA(vsd, intgroup = GROUP_COLUMN, returnData = TRUE)
percentVar <- round(100 * attr(pca, "percentVar"))
p_pca <- ggplot(pca, aes(PC1, PC2, color = .data[[GROUP_COLUMN]])) +
  geom_point(size = 3) +
  xlab(paste0("PC1: ", percentVar[1], "%")) +
  ylab(paste0("PC2: ", percentVar[2], "%")) +
  labs(color = GROUP_COLUMN, title = paste(PROJECT_ID, "PCA"))
ggsave(file.path(OUTDIR, "PCA.pdf"), p_pca, width = 6, height = 5)

for (comp_name in names(res_list)) {
  df <- res_list[[comp_name]] %>% filter(!is.na(padj))
  p <- EnhancedVolcano(df, lab = df$gene, x = "log2FoldChange", y = "padj",
                       pCutoff = PADJ_THRESH, FCcutoff = LOG2FC_THRESH, title = comp_name)
  ggsave(file.path(OUTDIR, paste0("Volcano_", comp_name, ".pdf")), p, width = 7, height = 8)
}


## 8. GEO Download Skeleton

In [ ]:
if (!is.null(GEO_ACCESSION)) {
  geo <- getGEO(GEO_ACCESSION, GSEMatrix = TRUE)
  eset <- geo[[1]]
  geo_expr <- exprs(eset)
  geo_pheno <- pData(eset)
  saveRDS(list(expr = geo_expr, pheno = geo_pheno), file.path(OUTDIR, paste0(GEO_ACCESSION, "_GEO.rds")))
  write.csv(geo_pheno, file.path(OUTDIR, paste0(GEO_ACCESSION, "_pheno.csv")), row.names = FALSE)
  cat("GEO loaded:", GEO_ACCESSION, dim(geo_expr)[1], "features x", dim(geo_expr)[2], "samples\n")
}


## 9. Single-Gene External Validation

In [ ]:
vsd_mat <- assay(vsd)
found <- intersect(TARGET_GENES, rownames(vsd_mat))
plot_df <- as.data.frame(t(vsd_mat[found, , drop = FALSE])) %>%
  rownames_to_column("sample") %>%
  left_join(metadata[, c("sample", GROUP_COLUMN)], by = "sample") %>%
  pivot_longer(cols = all_of(found), names_to = "gene", values_to = "expression")

if (nrow(plot_df) > 0) {
  p <- ggplot(plot_df, aes(x = .data[[GROUP_COLUMN]], y = expression, fill = .data[[GROUP_COLUMN]])) +
    geom_boxplot(outlier.shape = NA) +
    geom_jitter(width = 0.15, size = 1.5) +
    facet_wrap(~ gene, scales = "free_y") +
    labs(x = NULL, y = "VST expression", title = "Target gene validation")
  ggsave(file.path(OUTDIR, "Target_gene_expression.pdf"), p, width = 10, height = 5)
}
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
